### Creates `openalex.works.locations_w_sources` in Walden End to End workflow

In [ ]:
CREATE OR REPLACE TABLE openalex.works.locations_w_sources
CLUSTER BY (best_doi, provenance, native_id)
TBLPROPERTIES (
  'delta.dataSkippingNumIndexedCols' = 40,
  'delta.deletedFileRetentionDuration' = '14 days',
  'delta.logRetentionDuration' = '14 days'
)
AS (
with all_locations as (
  select
    *
  from
    openalex.works.superlocations
),
sources_filtered AS (
  SELECT *
  FROM openalex.sources.sources
  WHERE id IS NOT NULL  -- mirror is active-only post-D1
    AND id NOT IN (
      4377196106, -- Internet Archive (Internet Archive)
      4306400349, -- CiteSeer X (The Pennsylvania State University)
      4308707206, -- Library Union Catalog of Bavaria, Berlin and Brandenburg (B3Kat Repository)
      4377196246, -- CISION
      4306400063, -- Scopus (Elsevier)
      7407052528, -- ResearchGate
      4306402117, -- Portail des Bibliotheques Municipales Specialisees (City of Paris)
      7407051242, -- E-Pics Bildarchiv
      7407051288, -- WSL Landesforstinventar
      4306400108, -- Library Stack (Library Stack)
      4306525896  -- Research Square (duplicate of 4306402450)
    )
),
sources_intermediate as (
  select
    id,
    explode(issns) as issn,
    sample_pmh_record,
    is_oa as is_oa_source
  from
    sources_filtered
),
existing_sources_for_other_works as (
  select
    id,
    row_number() over (partition by issn order by id asc) as id_row_number,
    issn,
    is_oa_source,
    sample_pmh_record,
    split_part(sample_pmh_record, ":", 2) as pmh_record_matching_string
  from
    sources_intermediate
),
existing_sources_for_repos as (
  select
    s.id,
    e.endpoint_id,
    s.is_oa as is_oa_source,
    row_number() over (
        partition by e.endpoint_id
        order by s.id asc
      ) as repo_row_number
  from
    openalex.sources.endpoint_to_source e
    join sources_filtered s on e.source_id = s.id
),
existing_sources_for_datacite as (
  select
    id,
    exploded_datacite_id as datacite_id,
    is_oa as is_oa_source,
    row_number() over (partition by exploded_datacite_id order by id asc) as datacite_row_number
  from
    sources_filtered
    lateral view explode(datacite_ids) t as exploded_datacite_id
  where
    datacite_ids is not null 
    and size(datacite_ids) > 0
),
datacite_issn_journal_map as (
  -- DataCite container-ISSN tier (SPEC-v2 2026-08-12): fresh uniqueness-1 map of
  -- normalized ISSN -> live journal. Built from sources_filtered (mirror is
  -- active-only post-D1, so no merge filter -- and the excluded-ID denylist is
  -- already applied there). Deliberately NOT reusing existing_sources_for_other_works:
  -- its lowest-id-wins ranking resolves ambiguity, but an ISSN owned by two live
  -- journals must match NOTHING here (skip-on-ambiguity rule).
  select
    norm_issn,
    max(id) as journal_id,
    max(is_oa) as journal_is_oa
  from (
    select distinct
      s.id,
      s.is_oa,
      upper(concat(regexp_extract(e.issn, '(\\d{4})-?(\\d{3}[\\dXx])', 1), '-',
                   regexp_extract(e.issn, '(\\d{4})-?(\\d{3}[\\dXx])', 2))) as norm_issn
    from
      sources_filtered s
      lateral view explode(s.issns) e as issn
    where
      s.type = 'journal'
      -- Tier-local destination denylist (FINAL-GRADE-20260814 required fix 2),
      -- same pattern as the sources_filtered excluded-ID list above:
      and s.id not in (
        4306560351 -- High Technology Letters: its ISSN identity (1006-6748) is operated by a hijacked clone (gjstx-e.cn); the tier's 1,571-candidate cern.zenodo flow is the fake journal's content
      )
      and regexp_extract(e.issn, '(\\d{4})-?(\\d{3}[\\dXx])', 1) <> ''
  )
  group by
    norm_issn
  having
    count(distinct id) = 1
),
datacite_issn_exploded as (
  -- Container ISSNs a DataCite location declares in `ids`: namespace='issn' with an
  -- IsPartOf/IsPublishedIn relation. Ingest keeps `relationship` in verbatim depositor
  -- case, so compare case-insensitively. Normalization regex pinned to match the map.
  select
    native_id,
    lower(coalesce(raw_type, '')) as raw_type_norm,
    upper(concat(regexp_extract(container_issn.id, '(\\d{4})-?(\\d{3}[\\dXx])', 1), '-',
                 regexp_extract(container_issn.id, '(\\d{4})-?(\\d{3}[\\dXx])', 2))) as norm_issn
  from
    all_locations
    lateral view explode(
      filter(
        ids,
        x -> x.namespace = 'issn'
          and lower(x.relationship) in ('ispartof', 'ispublishedin')
          and regexp_extract(x.id, '(\\d{4})-?(\\d{3}[\\dXx])', 1) <> ''
      )
    ) t as container_issn
  where
    provenance = 'datacite'
),
datacite_issn_candidates as (
  -- Uniqueness rule (per LOCATION, over destinations): all qualifying container ISSNs
  -- of a location must resolve to EXACTLY ONE distinct live journal (print + e-ISSN of
  -- the same journal = attach; two distinct journals = no candidate). The group-by
  -- guarantees at most one candidate row per native_id.
  -- Genre guard (FINAL-GRADE-20260814 required fix 1): only text-like deposits may
  -- attach to a journal. raw_type carries DataCite's resourceTypeGeneral (lowercased;
  -- some ingest paths use CSL-style values like 'article-journal'). Allow-list built
  -- from the measured in-cohort distribution; everything else -- Dataset, Software,
  -- Image, Journal (journal-level records), ... -- is excluded: those are mostly
  -- supplementary deposits citing their parent article's journal via IsPartOf.
  -- 'peerreview' stays IN (mediarep encodes published book reviews as PeerReview).
  -- NULL raw_type ('' after coalesce) stays IN (measured: 1,007 rows, all
  -- Eurographics conference papers). 'other' stays OUT (sampled ~40%
  -- supplement/metadata deposits).
  select
    e.native_id,
    max(m.journal_id) as issn_journal_id,
    max(m.journal_is_oa) as issn_journal_is_oa
  from
    datacite_issn_exploded e
    join datacite_issn_journal_map m on m.norm_issn = e.norm_issn
  where
    e.raw_type_norm in (
      '', 'text', 'journalarticle', 'article-journal', 'article', 'preprint',
      'conferencepaper', 'paper-conference', 'conferenceproceeding', 'peerreview',
      'report', 'book', 'bookchapter', 'chapter', 'dissertation', 'thesis', 'datapaper'
    )
  group by
    e.native_id
  having
    count(distinct m.journal_id) = 1
),
datacite_issn_secondary_present as (
  -- Skip-when-present (SPEC-v2 step-0 decision 2; secondary-copy tripwire TRIPPED at
  -- 8.01%): drop candidates whose work already has ANOTHER location on the destination
  -- journal (typically a Zenodo copy of the published article -- moving it would
  -- duplicate, not correct). Work linkage does not exist at this stage of the E2E
  -- (works assembly runs after this notebook), so this reads the PRIOR RUN's
  -- works.locations_mapped as a snapshot: native_id -> work_id -> existing
  -- destination-journal location. This mirrors exactly how the validated shadow
  -- computed it; accepted staleness is one run -- a journal location first appearing
  -- in the current run is invisible here, so its work still moves this run and the
  -- skip takes effect from the next run onward. The candidate row ITSELF is excluded
  -- from the probe: once this tier moves a row, its own journal attachment appears in
  -- the next locations_mapped, and without self-exclusion every moved row would flip
  -- back on the following run.
  select distinct
    c.native_id
  from
    datacite_issn_candidates c
    join (
      select
        lower(native_id) as doi,
        max(work_id) as work_id
      from
        openalex.works.locations_mapped
      where
        provenance = 'datacite'
      group by
        lower(native_id)
    ) w on lower(c.native_id) = w.doi
    join openalex.works.locations_mapped lm
      on lm.work_id = w.work_id
      and lm.source_id = c.issn_journal_id
      and not (lm.provenance = 'datacite' and lower(lm.native_id) = w.doi)
    -- oxjob #1916: a sibling that is itself a candidate for the same journal is this tier's own output too, not an
    -- independent copy. Zenodo gives one deposit a concept DOI and a version DOI on the same work; each saw the other
    -- on the journal in the prior run and stepped back, the next run neither was there and both moved again, so ~415K
    -- DataCite locations (~205K works) swapped Zenodo <-> journal every night (and re-stamped). With siblings ignored
    -- the probe reads only evidence this tier does not write (Crossref, other feeds, DataCite records that are not
    -- candidates for that journal), so the answer is the same every night. Revert: delete this join and the WHERE below.
    left join datacite_issn_candidates sib
      on lm.provenance = 'datacite'
      and lower(lm.native_id) = lower(sib.native_id)
      and sib.issn_journal_id = c.issn_journal_id
  where
    sib.native_id is null
),
other_works as (
  select
    *
  from
    all_locations
  where
    provenance not in ('repo', 'repo_backfill', 'datacite', 'pubmed')
),
other_works_exploded as (
  select
    *,
    explode_outer(
      filter(
        ids,
        x -> contains(x.namespace, 'issn')
        and (
          x.relationship in (
            "self",
            "IsVersionOf",
            "IsNewVersionOf",
            "IsPreviousVersionOf",
            "IsVariantFormOf",
            "IsOriginalFormOf",
            "IsIdentialTo",
            "IsTranslationOf",
            "HasTranslation"
          )
          or x.relationship is null
        )
      )
    ) as relevant_id,
    relevant_id.namespace as namespace
  from
    other_works
),
other_works_ranked as (
  select
    *,
    row_number() over (
        partition by native_id
        order by
          case
            when namespace = 'eissn' then 1
            when namespace = 'lissn' then 2
            when namespace = 'pissn' then 3
          end,
          relevant_id.id
      ) as native_id_row_number
  from
    other_works_exploded
),
other_works_with_sources as (
  select
    native_id,
    id,
    is_oa_source,
    provenance,
    row_number() over (
        partition by native_id
        order by native_id_row_number asc
      ) as best_source_match
  from
    other_works_ranked
      left join existing_sources_for_other_works
        on other_works_ranked.relevant_id.id = existing_sources_for_other_works.issn
  where
    existing_sources_for_other_works.id is not null
  qualify
    best_source_match = 1
),
other_works_final as (
  select
    w.provenance,
    w.native_id,
    w.native_id_namespace,
    w.title,
    w.normalized_title,
    w.authors,
    w.ids,
    w.raw_type,
    w.type,
    w.version,
    w.license,
    w.language,
    w.published_date,
    w.created_date,
    w.updated_date,
    w.issue,
    w.volume,
    w.first_page,
    w.last_page,
    w.is_retracted,
    w.abstract,
    w.source_name,
    w.publisher,
    w.funders,
    w.references,
    w.urls,
    w.mesh,
    w.is_oa,
    w.endpoint_id,
    w.abstract_inverted_index,
    w.authors_exist,
    w.affiliations_exist,
    w.is_corresponding_exists,
    w.best_doi,
    w.merge_key,
    w.pdf_url,
    w.landing_page_url,
    w.pdf_s3_id,
    w.grobid_s3_id,
    w.ingested_at,
    ws.is_oa_source,
    ws.id as source_id
  from
    other_works w
      left join other_works_with_sources ws
        on w.native_id == ws.native_id
),
datacite_exploded as (
  select
    *,
    explode_outer(
      filter(
        ids,
        x -> x.namespace = 'datacite_client'
        and (x.relationship = 'self' or x.relationship is null)
      )
    ) as datacite_client_id
  from
    all_locations
  where
    provenance = "datacite"
),
datacite_with_sources as (
  select
    d.*,
    s.is_oa_source,
    s.id as source_id,
    row_number() over (
        partition by d.native_id
        order by s.id asc
      ) as best_datacite_match
  from
    datacite_exploded d
      left join existing_sources_for_datacite s
        on d.datacite_client_id.id = s.datacite_id
  qualify
    best_datacite_match = 1 or best_datacite_match is null
),
datacite_with_issn_tier as (
  -- Container-ISSN tier application + precedence (SPEC-v2, decided): the ISSN
  -- attachment applies ONLY where the datacite_client tier left the row NULL or on a
  -- repository-type source. A client resolving to a JOURNAL is registry-curated,
  -- depositor-verified evidence and WINS -- rows whose container ISSN disagrees with a
  -- journal-resolved client (the 537-row conflict class) are left untouched here and
  -- reviewed offline (no review-table write in the prod notebook). Because this sits
  -- BEFORE the union, the tier outranks every shared fallback below (display-name /
  -- URL / DOI-prefix / publisher) and the post-CTAS MERGE cells -- all of those only
  -- touch source_id IS NULL rows.
  select
    d.* except (is_oa_source, source_id),
    case
      when c.native_id is not null
        and sk.native_id is null
        and (d.source_id is null or cur.type = 'repository')
      then c.issn_journal_is_oa
      else d.is_oa_source
    end as is_oa_source,
    case
      when c.native_id is not null
        and sk.native_id is null
        and (d.source_id is null or cur.type = 'repository')
      then c.issn_journal_id
      else d.source_id
    end as source_id
  from
    datacite_with_sources d
    left join datacite_issn_candidates c on c.native_id = d.native_id
    left join datacite_issn_secondary_present sk on sk.native_id = d.native_id
    left join sources_filtered cur on cur.id = d.source_id
),
datacite_final as (
  select
    provenance,
    native_id,
    native_id_namespace,
    title,
    normalized_title,
    authors,
    ids,
    raw_type,
    type,
    version,
    license,
    language,
    published_date,
    created_date,
    updated_date,
    issue,
    volume,
    first_page,
    last_page,
    is_retracted,
    abstract,
    source_name,
    publisher,
    funders,
    references,
    urls,
    mesh,
    is_oa,
    endpoint_id,
    abstract_inverted_index,
    authors_exist,
    affiliations_exist,
    is_corresponding_exists,
    best_doi,
    merge_key,
    pdf_url,
    landing_page_url,
    pdf_s3_id,
    grobid_s3_id,
    ingested_at,
    is_oa_source,
    source_id
  from
    datacite_with_issn_tier
),
pubmed_final as (
  select
    *,
    False as is_oa_source,
    4306525036 as source_id
  from
    all_locations
  where
    provenance = 'pubmed'
),
repo as (
  select
    *
  from
    all_locations
  where
    provenance in ('repo', 'repo_backfill')
),
repo_sources as (
  select
    *
  from
    existing_sources_for_repos
  where
    repo_row_number = 1
),
aggregator_overrides as (
  select endpoint_id, key_namespace, key_prefix, source_id as override_source_id, rule_id
  from openalex.sources.aggregator_source_overrides
  where status = 'active'
),
repo_keyed as (
  -- parse the record key once; NULL when the native_id has no recognised shape
  select
    r.*,
    case
      when r.native_id rlike '^[^:]+:oai:[^:]+:'          then regexp_extract(r.native_id, '^[^:]+:oai:([^:]+):', 1)  -- NARCIS  <prefix>:oai:<ns>:<id>
      when r.native_id rlike '^oai:irdb\\.nii\\.ac\\.jp:[0-9]+:' then 'irdb.nii.ac.jp'                                 -- IRDB    oai:irdb.nii.ac.jp:<code>:<id>
      when r.native_id rlike '^oai:[^:]+:'                then regexp_extract(r.native_id, '^oai:([^:]+):', 1)          -- plain   oai:<ns>:<id>
    end as key_namespace,
    case
      when r.native_id rlike '^[^:]+:oai:[^:]+:'          then regexp_extract(r.native_id, '^([^:]+):oai:', 1)
      when r.native_id rlike '^oai:irdb\\.nii\\.ac\\.jp:[0-9]+:' then regexp_extract(r.native_id, '^oai:irdb\\.nii\\.ac\\.jp:([0-9]+):', 1)
      else ''
    end as key_prefix
  from repo r
),
repo_final as (
  select
    r.provenance, r.native_id, r.native_id_namespace, r.title, r.normalized_title, r.authors, r.ids, r.raw_type, r.type,
    r.version, r.license, r.language, r.published_date, r.created_date, r.updated_date, r.issue, r.volume, r.first_page,
    r.last_page, r.is_retracted, r.abstract, r.source_name, r.publisher, r.funders, r.references, r.urls, r.mesh, r.is_oa,
    r.endpoint_id, r.abstract_inverted_index, r.authors_exist, r.affiliations_exist, r.is_corresponding_exists, r.best_doi,
    r.merge_key, r.pdf_url, r.landing_page_url, r.pdf_s3_id, r.grobid_s3_id, r.ingested_at,
    coalesce(os.is_oa, s.is_oa_source) as is_oa_source,
    coalesce(o.override_source_id, s.id) as source_id
    -- audit (add to the table only if the column list downstream tolerates it; otherwise keep in a side table in dev):
    -- , case when o.override_source_id is not null then o.rule_id else 'endpoint' end as source_rule
  from
    repo_keyed r
    left join repo_sources s on r.endpoint_id = s.endpoint_id
    left join aggregator_overrides o
      on o.endpoint_id = r.endpoint_id and o.key_namespace = r.key_namespace and o.key_prefix = r.key_prefix
    left join sources_filtered os on os.id = o.override_source_id
),
-- NOTE: unresolved NARCIS rows keep the endpoint's source. Repointing endpoint bfa41bc4dfeca7f7fae from DANS to a new
-- "NARCIS (historical)" source is a REGISTRY change (source_endpoint),
unioned as (
  select
    *
  from
    repo_final
  union
  select
    *
  from
    datacite_final
  union
  select
    *
  from
    pubmed_final
  union
  select
    *
  from
    other_works_final
),
-- Everything below until the "DISPLAY_NAME END COMMENT" is to match on display_name <--> source_name for non-matched locations.
sources_unique_display_names as (
  select
    display_name,
    max(id) as id,
    max(is_oa) as is_oa_source,
    max(coalesce(size(issns), 0)) > 0 as has_issns,
    count(*) as rwcnt
  from
    openalex.sources.sources
  group by
    display_name
  having
    rwcnt = 1 -- only keep display names if there is only one record for that display name
),
sources_unique_lower_display_names as (  -- oxjob #1891: the case-insensitive fallback in the match below
  select
    lower(display_name) as lower_display_name,
    max(id) as id,
    max(is_oa) as is_oa_source,
    max(coalesce(size(issns), 0)) > 0 as has_issns,
    count(*) as rwcnt
  from
    openalex.sources.sources
  group by
    lower(display_name)
  having
    rwcnt = 1
),
matched_records as (
  select
    *
  from
    unioned
  where
    source_id is not null
),
unmatched_records as (
  select
    *,
    -- match_name (oxjob #1891): the source name with control characters cleaned the way the registry cleans source
    -- names (openalex-sources sources_lib.clean_control_chars, decode_mojibake=False: MARC non-sort marks dropped, a
    -- lone Windows-1252 quote or dash mapped, tabs/newlines to spaces, other controls dropped, spaces collapsed).
    -- Raw names from library-catalogue imports carry the marks ('\x98The \x9cNew York times book review'); the
    -- registry dropped them from 1,635 display names on 27 Oct, so the exact match compares cleaned to cleaned.
    -- A name without a control character is left exactly as it is.
    case
      when source_name rlike '[\\x{0000}-\\x{001F}\\x{007F}-\\x{009F}]' then
        trim(regexp_replace(regexp_replace(regexp_replace(
          translate(regexp_replace(source_name, '[\\x{0088}\\x{0089}\\x{0098}\\x{009C}]', ''),
            '\u0082\u0084\u0085\u008B\u0091\u0092\u0093\u0094\u0095\u0096\u0097\u009B',
            '\u201A\u201E\u2026\u2039\u2018\u2019\u201C\u201D\u2022\u2013\u2014\u203A'),
          '[\\t\\n\\r\\x{000B}\\x{000C}]', ' '), '[\\x{0000}-\\x{001F}\\x{007F}-\\x{009F}]', ''), '(?U)\\s+', ' '))
      else source_name
    end as match_name
  from
    unioned
  where
    source_id is null
),
try_display_name_match_on_unmatched_records as (
  select
    * except (a.source_id, a.is_oa_source, a.match_name, b.display_name, b.id, b.rwcnt, b.has_issns, b.is_oa_source,
              c.lower_display_name, c.id, c.rwcnt, c.has_issns, c.is_oa_source),
    coalesce(b.is_oa_source, c.is_oa_source) as is_oa_source,
    coalesce(b.id, c.id) as source_id
  from
    unmatched_records a
      left join sources_unique_display_names b
        on a.match_name = b.display_name
        -- A book or chapter's container title is the book's title; it may only name a source with no ISSNs
        -- (oxjob #828: CRC "Food Safety" chapters on the journal Food Safety). Same type list as cell 3.
        and not (coalesce(lower(a.raw_type) in (
        'book', 'books', 'book-chapter', 'book chapter', 'bookchapter', 'ebook chapter',
        'book-part', 'book part', 'bookpart', 'book section', 'book-section', 'book article', 'chapter', 'inbook',
        'monograph', 'printed monograph', 'edited-book', 'reference-book', 'book-set',
        'libros', 'libro', 'info:eu-repo/semantics/book', 'info:eu-repo/semantics/bookpart'
        ), false) and b.has_issns)
      left join sources_unique_lower_display_names c
        -- Case-insensitive fallback, only for names that had a control character (oxjob #1891): the registry gave 44
        -- cleaned names Crossref's casing ('\x98The \x9ccryosphere' -> 'The Cryosphere'), so the cleaned raw name
        -- differs from them in case only. Same book guard.
        on b.id is null
        and a.match_name <> a.source_name
        and lower(a.match_name) = c.lower_display_name
        and not (coalesce(lower(a.raw_type) in (
        'book', 'books', 'book-chapter', 'book chapter', 'bookchapter', 'ebook chapter',
        'book-part', 'book part', 'bookpart', 'book section', 'book-section', 'book article', 'chapter', 'inbook',
        'monograph', 'printed monograph', 'edited-book', 'reference-book', 'book-set',
        'libros', 'libro', 'info:eu-repo/semantics/book', 'info:eu-repo/semantics/bookpart'
        ), false) and c.has_issns)
),
unioned_with_display_name_matches as (
  select * from matched_records union select * from try_display_name_match_on_unmatched_records
),
-- DISPLAY_NAME END COMMENT of matching on display_name <--> source_name for unmatched locations.
-- manual matching by varios methods
matched_records_2 as (
  select
    *
  from
    unioned_with_display_name_matches
  where
    source_id is not null
),
unmatched_records_2 as (
  select
    *,
    REGEXP_EXTRACT(coalesce(landing_page_url, pdf_url), 'https?://(www\\.)?([^/]+)', 2) AS extracted_base_url
  from
    unioned_with_display_name_matches
  where
    source_id is null
),
-- by url
manual_url_matching as (
  select * except(extracted_base_url, source_id), 
  case 
    -- no rules for ci.nii.ac.jp, cqvip.com, books.openedition.org: they sent every unmatched record on those hosts
    -- to one journal (2.9M CiNii catalogue records on Medical Entomology and Zoology; oxjob #1225, ticket 26471)
    when extracted_base_url = 'europepmc.org' then 4306400806
    when extracted_base_url = 'dialnet.unirioja.es' then 4306401293
    when extracted_base_url = 'osti.gov' then 4306402487
    when extracted_base_url = "cyberleninka.ru" then 4306401404
    when extracted_base_url = "ntrs.nasa.gov" then 4306402118
    when extracted_base_url = "scopus.com" then 4306400063
    when contains(coalesce(landing_page_url, pdf_url), "hal.science") then 4306402512
    else source_id 
  end as source_id
  from unmatched_records_2
),
-- by doi prefix
manual_doi_matching as (
  select * except(source_id), 
  case 
    when best_doi like '10.20944/preprints%' then 6309402219
    when best_doi like '10.2139/ssrn.%' then 4210172589
    when best_doi like '10.26434/chemrxiv%' then 4393918830
    else source_id
  end as source_id
  from manual_url_matching
),
-- by source_name
manual_source_name_matching as (
  select * except(source_id), 
  case
    when LOWER(source_name) = 'biorxiv' then 4306402567
    when LOWER(source_name) = 'medrxiv' then 3005729997
    when LOWER(source_name) = 'research square' then 4306402450
    else source_id
  end as source_id
  from manual_doi_matching
),
-- by publisher
manual_publisher_matching as (
  select * except(source_id), 
  case
    when LOWER(publisher) like '%research square platform%' then 4306402450
    else source_id
  end as source_id
  from manual_source_name_matching
),
unioned_with_all_matching as (
  select * from matched_records_2 
  union 
  select * from manual_publisher_matching  -- includes everything from previous steps
)
select * from unioned_with_all_matching
);

### Create and populate normalized columns

In [0]:
ALTER TABLE openalex.works.locations_w_sources
  ADD COLUMN normalized_source_name STRING AFTER source_name;

ALTER TABLE openalex.works.locations_w_sources
  ADD COLUMN normalized_book_publisher STRING AFTER publisher;

UPDATE openalex.works.locations_w_sources
  SET 
    normalized_source_name = LOWER(REGEXP_REPLACE(source_name, '[^\\p{L}\\p{N}]', '')),
    normalized_book_publisher = CASE 
      -- ingest `type` is being nulled (cascade owns type); match the source-native raw type
      -- strings instead (measured: crossref/datacite enums + repo eu-repo URIs + top text variants)
      WHEN LOWER(raw_type) IN (
        'book', 'books', 'book-chapter', 'book chapter', 'bookchapter', 'ebook chapter',
        'book-part', 'book part', 'bookpart', 'book section', 'book-section', 'book article', 'chapter', 'inbook',
        'monograph', 'printed monograph', 'edited-book', 'reference-book', 'book-set',
        'libros', 'libro', 'info:eu-repo/semantics/book', 'info:eu-repo/semantics/bookpart'
      ) AND publisher IS NOT NULL 
      THEN LOWER(REGEXP_REPLACE(publisher, '[^\\p{L}\\p{N}]', ''))
      ELSE NULL 
    END;

### ERS congress abstracts: the European Respiratory Journal, by DOI (before every other tier)

In [0]:
-- ERS congress-abstract tier (oxjob #1528). The European Respiratory Society deposits its congress abstracts in Crossref as
-- proceedings-article with the SESSION as container title ("13.01 - Pulmonary hypertension", "Epidemiology", "Lung
-- Cancer") and no ISSN, so the name tier in cell 1 found nothing (40.5K) or a same-named journal or session record (5.8K:
-- Elsevier's Lung Cancer, LWW's Epidemiology...). They are published as the European Respiratory Journal's congress
-- supplement (vol. 54 suppl. 63 = Congress 2019), and the DOI carries ERJ's e-ISSN 1399-3003: 10.1183/13993003.congress-*
-- (10.1183/1393003.congress-2017.* is ERS's own typo of it). Proceedings inside a serial with an ISSN take the serial
-- (oxjob #1528 POLICY.md, 2026-10-03). Bar: repository copies of these DOIs name ERJ in 1,254 of 1,283 venue strings
-- that name a journal (LB 0.968). Runs before every other tier cell and OVERRIDES cell 1's name match (the tiers below
-- fill NULL sources only). Never touches a deposit that carries an ISSN. Revert: delete this cell.
MERGE INTO openalex.works.locations_w_sources AS target
USING (
  SELECT id AS source_id, is_oa AS is_oa_source
  FROM openalex.sources.sources
  WHERE id = 4210207896 -- European Respiratory Journal (ISSN-L 0903-1936)
) AS source
ON target.provenance = 'crossref'
   AND LOWER(target.raw_type) = 'proceedings-article'
   AND (LOWER(target.best_doi) LIKE '10.1183/13993003.congress-%' OR LOWER(target.best_doi) LIKE '10.1183/1393003.congress-%')
   AND NOT exists(target.ids, x -> x.namespace LIKE '%issn%')
   AND (target.source_id IS NULL OR target.source_id <> source.source_id)
WHEN MATCHED THEN
  UPDATE SET target.source_id = source.source_id, target.is_oa_source = source.is_oa_source;

### ATS conference abstracts: the American Journal of Respiratory and Critical Care Medicine, by DOI (before every other tier)

In [0]:
-- ATS conference-abstract tier (oxjob #1528). The American Thoracic Society deposits its International Conference abstracts
-- in Crossref as proceedings-article with the SESSION as container title ("A32. ASTHMA: CLINICAL STUDIES II") and no ISSN,
-- so no tier found a source (56.7K on 1 Nov 2026). They are published as the American Journal of Respiratory and Critical
-- Care Medicine's meeting-abstracts supplement (vol. 199 Supplement_1 = 2019, page A1308), and the DOI carries the journal
-- and its volume: 10.1164/ajrccm-conference.<year>.<volume>.1_meetingabstracts.<page>. Proceedings inside a serial with an
-- ISSN take the serial (oxjob #1528 POLICY.md, 2026-10-03; same as the ERS cell above). Bars: citing papers name AJRCCM in
-- 2,863 of 2,986 reference strings that name a journal; the journal's platform (Oxford Academic) lists 150 of 150 sampled
-- DOIs in an AJRCCM Supplement. Same semantics as the ERS cell: before every other tier, overrides a name match. Never
-- touches a deposit that carries an ISSN, nor the session-level 'proceedings' records. Revert: delete this cell.
MERGE INTO openalex.works.locations_w_sources AS target
USING (
  SELECT id AS source_id, is_oa AS is_oa_source
  FROM openalex.sources.sources
  WHERE id = 93356601 AND type = 'journal' -- American Journal of Respiratory and Critical Care Medicine (ISSN-L 1073-449X)
) AS source
ON target.provenance = 'crossref'
   AND LOWER(target.raw_type) = 'proceedings-article'
   AND LOWER(target.best_doi) LIKE '10.1164/ajrccm-conference.%'
   AND NOT exists(target.ids, x -> x.namespace LIKE '%issn%')
   AND (target.source_id IS NULL OR target.source_id <> source.source_id)
WHEN MATCHED THEN
  UPDATE SET target.source_id = source.source_id, target.is_oa_source = source.is_oa_source;

### BMJ meeting abstracts: the journal the DOI names (before every other tier)

In [0]:
-- BMJ meeting-abstract tier (oxjob #1528). BMJ deposits the meeting abstracts its journals publish in supplements (BTS in
-- Thorax, IDDF in Gut, SITC in JITC, EAHP in EJHP ...) as proceedings-article with a session or generic heading as
-- container title ("Abstracts", "Poster presentations") and no ISSN, so they had no source (~54K) or sat on session-name
-- records (~23K) on 1 Nov 2026. The DOI names the journal: 10.1136/<journal code>-<year>-<meeting>.<n> (old style
-- 10.1136/gut.2009.209080y). Proceedings inside a serial with an ISSN take the serial (oxjob #1528 POLICY.md, 2026-10-03;
-- same as the ERS and ATS cells above). Code -> journal: the journal holding that code's own journal-article DOIs; a code
-- is in only when 5 of 5 sampled abstracts land (doi.org) on that journal's own site. Bar: 150 of 150 fresh samples land
-- there. ADC's code (archdischild) is out: its main, Fetal & Neonatal and Education & Practice editions share it.
-- Same semantics as the ERS cell: before every other tier, overrides a name match. Index pages (.authorindex ...) and
-- deposits that carry an ISSN are never touched. Revert: delete this cell.
MERGE INTO openalex.works.locations_w_sources AS target
USING (
  SELECT m.code, s.id AS source_id, s.is_oa AS is_oa_source
  FROM (VALUES
    ('gutjnl', 3251770),
    ('jitc', 2754803477),
    ('ejhpharm', 3012707348),
    ('heartjnl', 191565391),
    ('spcare', 4210204300),
    ('oemed', 159252376),
    ('thorax', 106546978),
    ('rapm', 3341432),
    ('lupus', 2764956401),
    ('sextrans', 112197712),
    ('injuryprev', 81674035),
    ('jnis', 66405466),
    ('jech', 156988948),
    ('jnnp', 134773490),
    ('bmjspcare', 4210204300),
    ('neurintsurg', 66405466),
    ('bjsports', 136578784),
    ('bmjpo', 4306504555),
    ('bmjstel', 2764470026),
    ('bmjno', 4306504553),
    ('annrheumdis', 25650217),
    ('injuryprevention', 81674035),
    ('oem', 159252376),
    ('leader', 4306504552),
    ('bmjopen', 79054089),
    ('flgastro', 125615467),
    ('bmjgh', 2764928273),
    ('thoraxjnl', 106546978),
    ('bmjophth', 4210181307),
    ('bmjresp', 2736820664),
    ('gut', 3251770),
    ('emermed', 127305661),
    ('bmjsem', 2764939095),
    ('ijgc', 41026820),
    ('bmjnph', 4210221727),
    ('qshc', 152101599),
    ('hrt', 191565391),
    ('openhrt', 4210199616),
    ('bmjgast', 2764956485),
    ('jmedgenet', 112540174),
    ('emj', 127305661),
    ('jmg', 112540174),
    ('bmjsit', 4210223288),
    ('postgradmedj', 43893605),
    ('bmjhci', 4210217889)
  ) AS m(code, source_id)
  JOIN openalex.sources.sources s ON s.id = m.source_id AND s.type = 'journal'
) AS source
ON target.provenance = 'crossref'
   AND LOWER(target.raw_type) = 'proceedings-article'
   AND target.best_doi LIKE '10.1136/%'
   AND LOWER(regexp_extract(target.best_doi, '^10\\.1136/([a-zA-Z]+)', 1)) = source.code
   AND NOT LOWER(target.best_doi) RLIKE '\\.(authorindex|subjectindex|programme|program|introduction|committees|editorial|intro|fmatter|contents)$'
   AND NOT exists(target.ids, x -> x.namespace LIKE '%issn%')
   AND (target.source_id IS NULL OR target.source_id <> source.source_id)
WHEN MATCHED THEN
  UPDATE SET target.source_id = source.source_id, target.is_oa_source = source.is_oa_source;

### Match book-typed locations to their publisher's ebook platform

In [0]:
-- Ebook-platform tier (oxjob #1528). A book or chapter's source is its publisher's ebook platform.
-- Key: the location's normalized publisher equals a platform's display_name or one of its alternate_titles,
-- minus a trailing "eBooks" and normalized the same way. The registry stores publisher-string aliases on
-- each platform in that form ("Springer Berlin Heidelberg eBooks" on Springer eBooks). Unique keys only:
-- a key that names two platforms matches nothing.
WITH ebook_platform_titles AS (
  SELECT id AS source_id,
    LOWER(REGEXP_REPLACE(REGEXP_REPLACE(t, '(?i)\\s*ebooks$', ''), '[^\\p{L}\\p{N}]', '')) AS k
  FROM openalex.sources.sources
  LATERAL VIEW explode(concat(array(display_name), coalesce(alternate_titles, array()))) x AS t
  WHERE type = 'ebook platform' AND t IS NOT NULL
),
ebook_platform_keys AS (
  SELECT k, MIN(source_id) AS source_id
  FROM ebook_platform_titles
  WHERE length(k) >= 3
  GROUP BY k
  HAVING COUNT(DISTINCT source_id) = 1
)
MERGE INTO openalex.works.locations_w_sources AS target
USING ebook_platform_keys AS source
ON target.source_id IS NULL
   AND target.normalized_book_publisher IS NOT NULL
   AND target.normalized_book_publisher = source.k
WHEN MATCHED THEN
  UPDATE SET target.source_id = source.source_id;

-- Older key, kept for what the name key misses: the platform's own publisher column, unique among platforms.
WITH ebook_platform_sources AS (
  SELECT
    id AS source_id,
    LOWER(REGEXP_REPLACE(publisher, '[^\\p{L}\\p{N}]', '')) AS normalized_publisher,
    COUNT(*) OVER (
      PARTITION BY LOWER(REGEXP_REPLACE(publisher, '[^\\p{L}\\p{N}]', ''))
    ) AS name_count
  FROM openalex.sources.sources
  WHERE type = 'ebook platform'
    AND publisher IS NOT NULL
  QUALIFY name_count = 1
)
MERGE INTO openalex.works.locations_w_sources AS target
USING ebook_platform_sources AS source
ON target.source_id IS NULL
   AND target.normalized_book_publisher IS NOT NULL
   AND target.normalized_book_publisher = source.normalized_publisher
WHEN MATCHED THEN
  UPDATE SET target.source_id = source.source_id;

### Match proceedings papers to their conference series

In [0]:
-- Conference-series tier (oxjob #1528). A proceedings paper's source is its conference SERIES, not one year's
-- edition. Crossref names editions ("ICASSP 2026 - 2026 IEEE International Conference on ..."); the series key
-- drops years, ordinals, "Proceedings of", bracketed acronyms and non-letters, so every edition of a series
-- shares one key. openalex.sources.conference_series_keys maps judged keys (one series each, checked per
-- cluster) to their series source; only keys in that table match, so an unreviewed conference record
-- can't capture papers that merely share its key.
MERGE INTO openalex.works.locations_w_sources AS target
USING (
  SELECT k.series_key, k.source_id
  FROM openalex.sources.conference_series_keys k
  JOIN openalex.sources.sources s ON s.id = k.source_id AND s.type = 'conference'
) AS source
ON target.source_id IS NULL
   AND LOWER(target.raw_type) IN ('proceedings-article', 'proceedings')
   AND target.source_name IS NOT NULL
   AND regexp_replace(
      regexp_replace(
        regexp_replace(
          regexp_replace(
            regexp_replace(lower(replace(target.source_name, '&amp;', '&')), '\\([^)]*\\)|\\[[^\\]]*\\]', ' '),
            '^[^-–—:]{0,60}\\b(19|20)[0-9]{2}\\b[^-–—:]{0,15}\\s[-–—:]\\s', ' '),
          '\\b(proceedings|proceeding|proc|of the|annual|[0-9]+(st|nd|rd|th)?|first|second|third|fourth|fifth|sixth|seventh|eighth|ninth|tenth|eleventh|twelfth)\\b', ' '),
        '\'[0-9]{2}', ' '),
      '[^\\p{L}]', '') = source.series_key
WHEN MATCHED THEN
  UPDATE SET target.source_id = source.source_id;

### Match normalized source names to source display names as final fallback

In [0]:
WITH final_display_name_fallback_sources AS ( -- get only unique titles (idea for later - check if it is non-numeric, length > N, more sources dedup coming)
  SELECT
    id AS source_id,
    issns,
    LOWER(REGEXP_REPLACE(display_name, '[^\\p{L}\\p{N}]', '')) AS normalized_source_name,
    COUNT(*) OVER (
      PARTITION BY LOWER(REGEXP_REPLACE(display_name, '[^\\p{L}\\p{N}]', ''))
    ) AS name_count
  FROM openalex.sources.sources
  WHERE id NOT IN ( -- repetitive - integrate this whole things into overall SQL later
      4377196106, -- Internet Archive (Internet Archive)
      4306400349, -- CiteSeer X (The Pennsylvania State University)
      4308707206, -- Library Union Catalog of Bavaria, Berlin and Brandenburg (B3Kat Repository)
      4377196246, -- CISION
      4306400063, -- Scopus (Elsevier)
      7407052528, -- ResearchGate
      4306402117, -- Portail des Bibliotheques Municipales Specialisees (City of Paris)
      7407051242, -- E-Pics Bildarchiv
      7407051288, -- WSL Landesforstinventar
      4306400108  -- Library Stack (Library Stack)
    )  
  QUALIFY name_count = 1
)
MERGE INTO openalex.works.locations_w_sources AS target
USING final_display_name_fallback_sources AS source
ON target.source_id IS NULL
   AND target.normalized_source_name = source.normalized_source_name
   -- same guard as cell 1 (oxjob #828): book-typed locations don't title-match ISSN-bearing sources
   AND NOT (COALESCE(LOWER(target.raw_type) IN (
     'book', 'books', 'book-chapter', 'book chapter', 'bookchapter', 'ebook chapter',
     'book-part', 'book part', 'bookpart', 'book section', 'book-section', 'book article', 'chapter', 'inbook',
     'monograph', 'printed monograph', 'edited-book', 'reference-book', 'book-set',
     'libros', 'libro', 'info:eu-repo/semantics/book', 'info:eu-repo/semantics/bookpart'
   ), FALSE) AND COALESCE(size(source.issns), 0) > 0)
WHEN MATCHED THEN
  UPDATE SET target.source_id = source.source_id;

### Match names of merged-away sources to their live winner

In [ ]:
-- Merged-name tier (oxjob #1528). A merge deletes the loser record (D1), and the name tiers above (cells 1 and 9) read
-- live display names only, so a location whose container name equals a merged-away record's name was left with no source:
-- 4.34M locations on 30 Oct 2026, nearly all MAG, mostly from the July 2026 dedup waves (日本泌尿器科學會雜誌 -> The Japanese
-- Journal of Urology, MPRA Paper -> Munich Personal RePEc Archive). openalex.sources.merged_source_names keeps the losers'
-- names (from the merge ledger, else the 7 Jul 2026 registry copy) and, per name, the location provenances whose merge
-- class passed the bar (tier_provenances; oxjob #1528 PLAN.md § Merged-name tier: MAG for the July dedup names, Crossref
-- for every class; MAG papers on this job's conference merges failed it and stay out). Such a location takes the loser's
-- live winner, as a stored loser id does (oxjob #1124), when every merged-away record with that normalized name has the
-- same live winner and no live source has that name (then cells 1 and 9 decide it, or it is ambiguous). Fills NULL
-- sources only. Same book guard as cells 1 and 9.
WITH live_names AS (
  SELECT DISTINCT LOWER(REGEXP_REPLACE(display_name, '[^\\p{L}\\p{N}]', '')) AS normalized_source_name
  FROM openalex.sources.sources
),
merged_names AS (
  SELECT
    LOWER(REGEXP_REPLACE(n.loser_name, '[^\\p{L}\\p{N}]', '')) AS normalized_source_name,
    MAX(s.id) AS source_id,
    MAX(COALESCE(size(s.issns), 0)) > 0 AS has_issns,
    COUNT(DISTINCT s.id) AS winners,
    array_distinct(flatten(collect_list(n.tier_provenances))) AS tier_provenances
  FROM openalex.sources.merged_source_names n
  JOIN openalex.sources.merged_source_ids m ON m.id = n.loser_id
  JOIN openalex.sources.sources s ON s.id = m.merge_into_id
  WHERE n.loser_name IS NOT NULL
  GROUP BY 1
)
MERGE INTO openalex.works.locations_w_sources AS target
USING (
  SELECT mn.normalized_source_name, mn.source_id, mn.has_issns, mn.tier_provenances
  FROM merged_names mn
  LEFT ANTI JOIN live_names l ON l.normalized_source_name = mn.normalized_source_name
  WHERE mn.winners = 1 AND size(mn.tier_provenances) > 0 AND mn.normalized_source_name <> ''
) AS source
ON target.source_id IS NULL
   AND target.normalized_source_name = source.normalized_source_name
   AND array_contains(source.tier_provenances, target.provenance)
   AND NOT (COALESCE(LOWER(target.raw_type) IN ('book', 'books', 'book-chapter', 'book chapter', 'bookchapter', 'ebook chapter',
     'book-part', 'book part', 'bookpart', 'book section', 'book-section', 'book article', 'chapter', 'inbook',
     'monograph', 'printed monograph', 'edited-book', 'reference-book', 'book-set',
     'libros', 'libro', 'info:eu-repo/semantics/book', 'info:eu-repo/semantics/bookpart'), FALSE) AND source.has_issns)
WHEN MATCHED THEN
  UPDATE SET target.source_id = source.source_id;

### Match peer reviews to the site that hosts them: the journal, or Publons / Web of Science

In [0]:
-- Peer-review tier (oxjob #1877). A Crossref peer review carries no journal (Crossref's peer-review record has no
-- container title or ISSN), so the tiers above leave it without a source. Its source is whoever hosts it, the party
-- that can take it down (Jason, 27 Oct), read from Crossref's registered landing page:
--   * on the same site as the article it reviews (relation is-review-of), registered by the same publisher: the
--     article's journal (eLife, PeerJ, F1000 platforms, Qeios, Copernicus journal sites, SciPost, Cambridge...);
--   * on Web of Science / Publons (most RSC, Wiley, Royal Society, OUP, SAGE, IOP transparent peer review): Publons.
-- Anything else keeps no source (EGUsphere-hosted Copernicus reviews: no EGUsphere source yet). Skip a review whose
-- articles sit in two different journals. Supplements (is-supplement-to) are left out on purpose for now.
-- Source stats leave peer reviews out of 2yr_mean_citedness (CreateSourcesApi, CreatePublishersAPI).
MERGE INTO openalex.works.locations_w_sources AS target
USING (
  WITH reviews AS (
    SELECT lower(c.native_id) AS doi, lower(r.id) AS article_doi
    FROM openalex.crossref.crossref_deduplicated c
    LATERAL VIEW explode(c.relation.`is-review-of`) x AS r
    WHERE r.`id-type` = 'doi' AND lower(r.id) <> lower(c.native_id)
  ),
  host_of AS (  -- registered landing-page host and Crossref member of every review and reviewed article
    SELECT lower(c.native_id) AS doi, c.member,
      lower(regexp_extract(c.resource.primary.URL, 'https?://(www\\.)?([^/]+)', 2)) AS host
    FROM openalex.crossref.crossref_deduplicated c
    LEFT SEMI JOIN (SELECT doi AS k FROM reviews UNION SELECT article_doi AS k FROM reviews) d
      ON d.k = lower(c.native_id)
  ),
  article_journal AS (
    SELECT lower(l.native_id) AS article_doi, max(l.source_id) AS source_id
    FROM openalex.works.locations_w_sources l
    JOIN openalex.sources.sources s ON s.id = l.source_id AND s.type = 'journal'
    WHERE l.provenance = 'crossref'
    GROUP BY lower(l.native_id)
    HAVING count(DISTINCT l.source_id) = 1
  ),
  on_journal_site AS (
    SELECT rv.doi, max(aj.source_id) AS source_id
    FROM reviews rv
    JOIN host_of h ON h.doi = rv.doi
    JOIN host_of ha ON ha.doi = rv.article_doi AND ha.host = h.host AND ha.member = h.member
    JOIN article_journal aj ON aj.article_doi = rv.article_doi
    GROUP BY rv.doi
    HAVING count(DISTINCT aj.source_id) = 1
  ),
  on_publons AS (
    SELECT DISTINCT rv.doi, CAST(4363604615 AS BIGINT) AS source_id  -- S4363604615 Publons reviews and discussion
    FROM reviews rv
    JOIN host_of h ON h.doi = rv.doi
    WHERE h.host IN ('webofscience.com', 'publons.com')
  )
  SELECT doi, source_id FROM on_journal_site
  UNION ALL
  SELECT p.doi, p.source_id FROM on_publons p LEFT ANTI JOIN on_journal_site j ON j.doi = p.doi
) AS source
ON target.source_id IS NULL
   AND target.provenance = 'crossref'
   AND lower(target.native_id) = source.doi
WHEN MATCHED THEN
  UPDATE SET target.source_id = source.source_id;

### ASME proceedings: the conference series named by the DOI (fills what every tier above left empty)

In [0]:
-- ASME conference-code tier (oxjob #1528). ASME deposits each conference volume with a volume label as container title
-- ("Volume 4: Heat Transfer, Parts A and B"), so the name and series-key tiers above can't tell which conference it is:
-- ~104K papers had no source on 31 Oct 2026. The DOI names it (10.1115/IMECE2011-62337, 10.1115/OMAE2009-80091), as does
-- Crossref's event name. The source is the conference SERIES (oxjob #1528 POLICY.md, 2026-10-03). One live series record
-- per code (GT and MSEC: their duplicate records are empty). FEDSM (joint years are the ASME/JSME Joint Fluids series)
-- and ICEF (two records) stay out. Bar: Crossref event names match the code's series on 300 of 300 sampled papers.
-- Fills NULL sources, and (ASME pass 2, 1 Nov 2026) takes these papers off 108 records they matched by name that are
-- not a conference of their own: volume labels ("Volume 1: Codes and Standards"), ASME division and volume titles typed
-- journal ("Advances in Bioengineering"), ISSN journals whose title equals a volume title (MDPI Materials, IEEE MEMS),
-- IMECE track titles minted as conferences ("Dynamic Systems and Control"), and another event (END 2022). Named
-- sub-conferences inside the umbrella (IDETC's Design Automation Conference, MESA, ...) keep their papers. Pass 2 bar:
-- Crossref event names match the code's series on 298 of 300 papers, 73-75 of 75 per class (list: #1528 conferences/asme/pass2_records.json).
-- Runs after every other tier, so it holds. Never touches a deposit that carries an ISSN.
-- Revert: delete this cell (pass 2 only: put the ON back to `target.source_id IS NULL`).
MERGE INTO openalex.works.locations_w_sources AS target
USING (
  SELECT m.code, s.id AS source_id, s.is_oa AS is_oa_source
  FROM (VALUES
    ('IMECE', 4363606942),
    ('DETC', 7407096944),
    ('OMAE', 4306499009),
    ('PVP', 4306503424),
    ('ICONE', 4210234662),
    ('DSCC', 4393918181),
    ('IPC', 4306497740),
    ('HT', 4306503359),
    ('SMASIS', 4306503494),
    ('ESDA', 4306503414),
    ('AJKFLUIDS', 4306503505),
    ('IPACK', 7407088218),
    ('ES', 4306503418),
    ('POWER', 4306503373),
    ('GTINDIA', 4363608538),
    ('ICEM', 4306503402),
    ('IJTC', 4306503509),
    ('MNHMT', 4306503412),
    ('GT', 4306503524),
    ('MSEC', 4306503479)
  ) AS m(code, source_id)
  JOIN openalex.sources.sources s ON s.id = m.source_id AND s.type = 'conference'
) AS source
ON (target.source_id IS NULL OR target.source_id IN (
       18714766, 62009534, 76910236, 184854148, 186735050, 1003244311, 1027733153, 4210169245,
       4210225457, 4210231368, 4210233918, 4306500777, 4306507803, 4306509109, 4306509110, 4306513397,
       4306534460, 4306534461, 4306534462, 4306534464, 4306534465, 4306534466, 4306534467, 4306534468,
       4306534471, 4306534472, 4306534474, 4306534475, 4306534476, 4306534477, 4306534478, 4306534479,
       4306534480, 4306534483, 4306534485, 4306534486, 4306534488, 4306534490, 4306534491, 4306534492,
       4306534493, 4306534494, 4306534495, 4306534497, 4306534498, 4306534499, 4306534500, 4306534505,
       4306534506, 4306534507, 4306534508, 4306534510, 4306534511, 4306534513, 4306534514, 4306534515,
       4306534518, 4306534519, 4306534520, 4306534521, 4306534525, 4306534526, 4306534527, 4306534528,
       4306534529, 4306534534, 4306534536, 4306534537, 4306534538, 4306534542, 4306534544, 4306534545,
       4306534546, 4306534547, 4363605141, 4363605687, 4363605754, 4363605757, 4363605837, 4363605972,
       4363606213, 4363606217, 4363606397, 4363606490, 4363606681, 4363607108, 4363607162, 4363607285,
       4363607352, 4363607513, 7407096277, 7407096794, 7407097217, 7407097517, 7407114468, 7407114563,
       7407119719, 7407120160, 7407120164, 7407120231, 7407120328, 7407120375, 7407120467, 7407120730,
       7407120804, 7407120830, 7407120939, 7407121113))
   AND target.provenance = 'crossref'
   AND LOWER(target.raw_type) = 'proceedings-article'
   AND target.best_doi LIKE '10.1115/%'
   AND NOT exists(target.ids, x -> x.namespace LIKE '%issn%')
   AND upper(regexp_extract(target.best_doi, '^10\\.1115/([a-zA-Z]+)[0-9]', 1)) = source.code
WHEN MATCHED THEN
  UPDATE SET target.source_id = source.source_id, target.is_oa_source = source.is_oa_source;

### IEEE proceedings: the conference series the DOI's acronym names (fills what every tier above left empty)

In [0]:
-- IEEE acronym tier (oxjob #1528). An IEEE conference paper's DOI carries the conference's IEEE acronym
-- (10.1109/icassp49357.2023.10095123, 10.1109/ieem.2007.4419132), but a container name the tiers above can't place leaves
-- it with no source (124K on 1 Nov 2026). Where IEEE papers with the same acronym already sit on one conference SERIES
-- (>= 50 papers, >= 95% on one record; oxjob #1528 POLICY.md, 2026-10-03: the source is the series), the acronym places
-- the rest. An acronym is in only when 3 of 3 sampled papers' Crossref event names (or, where the event name is a
-- fragment such as 'EM)', the container title) name that series; bar: 150 of 150 fresh samples do. Out: targets
-- named after one edition, IEEE's placeholder 'ieeeconf'. Fills NULL sources only and runs after every other tier. Never
-- touches a deposit that carries an ISSN. Revert: delete this cell.
MERGE INTO openalex.works.locations_w_sources AS target
USING (
  SELECT m.acr, s.id AS source_id, s.is_oa AS is_oa_source
  FROM (VALUES
    ('acirs', 7407088888),
    ('acssc', 4363608623),
    ('adcom', 7407088562),
    ('adl', 7407114315),
    ('afgr', 4306419159),
    ('aimlcps', 7407097394),
    ('apasic', 4306502575),
    ('apcc', 4363608215),
    ('apccas', 7407087888),
    ('apec', 4210177957),
    ('apeie', 4306498395),
    ('apemc', 4363605167),
    ('apmc', 4363608315),
    ('apmrc', 4306417760),
    ('aps', 4210177825),
    ('apsec', 4210216296),
    ('apwc', 7407088948),
    ('asap', 7407086867),
    ('asdam', 4306419106),
    ('ase', 4363608212),
    ('aswec', 4210205094),
    ('async', 4210190146),
    ('ats', 4210236090),
    ('avss', 7407086167),
    ('bcaa', 4306417672),
    ('bdai', 7407089195),
    ('bdva', 4306417833),
    ('beliv', 7407105861),
    ('bibe', 4393918218),
    ('camad', 7407087758),
    ('cbms', 4210229969),
    ('cca', 4210212887),
    ('ccece', 4210221841),
    ('ccgridw', 7407106335),
    ('ccnc', 4393918466),
    ('ccst', 7407088639),
    ('cdc', 4210221977),
    ('cec-east', 4306422628),
    ('cefc', 4363608280),
    ('ceidp', 4363608173),
    ('cem', 7407107329),
    ('cibcb', 7407088424),
    ('cicn', 4363608214),
    ('ciep', 7407087789),
    ('cies', 7407105601),
    ('cifer', 4306422520),
    ('cit', 7407085889),
    ('citcon', 4210197906),
    ('cleopr', 7407089323),
    ('cmpsac', 4210194295),
    ('cmv', 7407113694),
    ('cn', 7407113704),
    ('comite', 7407088494),
    ('comsnets', 7407088473),
    ('comswa', 4306417922),
    ('conect', 4210232710),
    ('conmedia', 7407088622),
    ('contel', 4363609009),
    ('cpem', 4306418090),
    ('cqr', 7407087892),
    ('csfw', 4210181444),
    ('csmr', 4210201492),
    ('ctc', 7407105705),
    ('cvpr', 4210176548),
    ('cwtm', 4306418797),
    ('dac', 4210231368),
    ('dcc', 4210220829),
    ('deiv', 4306420127),
    ('discex', 4306422023),
    ('disrta', 4210230007),
    ('distra', 4210230007),
    ('dspws', 7407096459),
    ('dtis', 7407085977),
    ('ease', 7407107833),
    ('ecbs', 7407086405),
    ('ecce-europe', 7407096092),
    ('ecceeurope', 7407096092),
    ('econf', 7407106578),
    ('edoc', 4210237449),
    ('edocw', 7407087000),
    ('eduhpc', 7407107643),
    ('eeei', 4306418611),
    ('eleco', 4363608342),
    ('electronica', 7407105081),
    ('elinsl', 4306418757),
    ('emceurope', 4363605869),
    ('empdp', 4210178956),
    ('empire', 7407107879),
    ('emrts', 4210175037),
    ('emwrts', 4210175037),
    ('enabl', 7407087244),
    ('enbeng', 4306422821),
    ('energycon', 4363608407),
    ('epep', 7407088465),
    ('eqec', 7407086845),
    ('esw', 7407086376),
    ('eurbot', 4306418332),
    ('eurdac', 4363608971),
    ('fie', 4363607793),
    ('fists', 7407106444),
    ('fpt', 7407087313),
    ('fuzz', 4210187232),
    ('glocom', 4210205019),
    ('glsv', 4363608736),
    ('gmap', 4306418480),
    ('group', 7407086434),
    ('hase', 4210186101),
    ('hdp', 4306420151),
    ('hfpsc', 4306418523),
    ('hicss', 4210195248),
    ('hipc', 4306418683),
    ('hipineb', 7407106239),
    ('hips', 4306418526),
    ('hkedm', 4306418628),
    ('holm', 7407089172),
    ('hotos', 4306421247),
    ('hpdc', 7407087987),
    ('ic-c', 7407105659),
    ('icacctech', 7407096490),
    ('icacit', 7407086780),
    ('icadeis', 7407086561),
    ('icairc', 7407096147),
    ('icaisisas', 4363607581),
    ('icalt', 4363608377),
    ('icamcs', 7407105596),
    ('icapp', 4306419122),
    ('icbdml', 4306423064),
    ('icc', 4306419201),
    ('icccas', 4306419202),
    ('icccn', 4210191573),
    ('iccdcs', 4306418967),
    ('iccoins', 4306423395),
    ('iccsa', 4306419223),
    ('iccsea', 4306498705),
    ('iccsit', 4306418665),
    ('iccsp', 7407088826),
    ('icctdc', 7407096207),
    ('iccv', 4363607764),
    ('iccw', 4393918252),
    ('icdcsw', 7407087786),
    ('icdse', 4306419326),
    ('icdsp', 7407086640),
    ('icecs', 4363607963),
    ('icelet', 7407106610),
    ('icetet', 4363608317),
    ('icetm', 7407088692),
    ('icfem', 4306419461),
    ('ichit', 4363608371),
    ('ichqp', 4306419496),
    ('ichr', 4363608580),
    ('iciap', 4306419518),
    ('iciasf', 7407087724),
    ('iciee', 7407106593),
    ('icieem', 4306418876),
    ('iciis', 7407088469),
    ('icimw', 4363607742),
    ('icmech', 7407088144),
    ('icmel', 7407088226),
    ('icmlc', 4306419646),
    ('icmse', 7407086177),
    ('icmts', 7407085999),
    ('icnic', 7407105875),
    ('icnp', 4210215197),
    ('icoac', 4306419096),
    ('icocs', 4306421317),
    ('iconstem', 7407086778),
    ('icosp', 4306419823),
    ('icpads', 7407087481),
    ('icpp', 4210174133),
    ('icpwc', 4306418698),
    ('icre', 7407086818),
    ('icrom', 4363608604),
    ('icse-seis', 7407086374),
    ('icsens', 4210192710),
    ('icsict', 4306418723),
    ('icslp', 4363607739),
    ('icsmc', 4210195764),
    ('icsr', 4306419846),
    ('icstc', 4306498266),
    ('ictai', 4210176007),
    ('icvd', 4306419912),
    ('iecec', 4306420377),
    ('iecon', 4210172894),
    ('iedm', 4210168112),
    ('ieem', 4363607822),
    ('iembs', 4363608786),
    ('iesc', 7407087337),
    ('igarss', 4363604196),
    ('iis', 7407088469),
    ('ijcnn', 4363607707),
    ('im', 4210179741),
    ('imoc', 4363608620),
    ('imsccs', 4306420027),
    ('incsst', 7407097112),
    ('intercon', 4306498332),
    ('iot', 7407087495),
    ('ipaps', 7407089254),
    ('ipds', 7407105211),
    ('iros', 4393918188),
    ('irps', 4210238090),
    ('irws', 4210230073),
    ('isads', 4306420098),
    ('isanp', 4306420091),
    ('isape', 4363604697),
    ('isbi', 4363605129),
    ('iscas', 4393914142),
    ('iscc', 4210175722),
    ('ise', 4306420130),
    ('isecon', 4363608318),
    ('isee', 4210182378),
    ('iseim', 7407087140),
    ('isie', 4363605663),
    ('isit', 4363604560),
    ('ismvl', 4210220027),
    ('isorc', 7407086058),
    ('ispsd', 4210228529),
    ('ispw', 4306420066),
    ('isqed', 4363607671),
    ('isrcs', 7407088705),
    ('isre', 7407086818),
    ('isscc', 4210210374),
    ('issre', 4210220341),
    ('issrew', 4363606990),
    ('isss', 7407086346),
    ('issse', 4306420213),
    ('isssta', 7407086186),
    ('istas', 7407088896),
    ('isvlsi', 4306418593),
    ('iswpc', 4306420237),
    ('itab', 4306418689),
    ('iteceats', 4306418838),
    ('itre', 4306419575),
    ('itsc', 4363607737),
    ('itw', 4363606450),
    ('ivelec', 7407087666),
    ('ivs', 4210198592),
    ('iwast', 7407087439),
    ('iwqos', 4210234387),
    ('iwrsp', 4210222258),
    ('iwsoc', 4306420356),
    ('iwstm', 7407113381),
    ('iwv', 4306418593),
    ('jcs', 7407104827),
    ('jcsse', 4306420003),
    ('jocici', 7407115708),
    ('kst', 4306419628),
    ('lanman', 7407086497),
    ('lcn', 4210182868),
    ('leosst', 4306518453),
    ('lescpe', 4306420442),
    ('lfnm', 4306419635),
    ('lics', 4210192682),
    ('lpe', 4363608740),
    ('lt', 7407088647),
    ('mahss', 7407086848),
    ('mascot', 4210229048),
    ('mcs', 7407087003),
    ('medhocnet', 4306417678),
    ('memcod', 7407088487),
    ('mems', 4210233918),
    ('mesoca', 7407113988),
    ('mhs', 4306420176),
    ('micro', 4210171338),
    ('miel', 7407088226),
    ('mmb', 4306497536),
    ('mmcs', 7407088426),
    ('mmee', 7407105376),
    ('mmet', 4306419658),
    ('mmse', 7407087905),
    ('mmsp', 4363605768),
    ('mosicom', 7407086287),
    ('mppoi', 7407113538),
    ('msmw', 4306424366),
    ('mtdt', 4210185863),
    ('mtv', 4210175880),
    ('mwp', 4306420247),
    ('mwscas', 4210192229),
    ('mwsym', 4210185519),
    ('nebc', 4393916914),
    ('neuront', 7407105581),
    ('nlo', 104011845),
    ('npsec', 7407105713),
    ('nrc', 4210215198),
    ('nrsc', 4306420582),
    ('olt', 7407087190),
    ('opnarc', 7407120741),
    ('orss', 7407107056),
    ('pact', 4210194525),
    ('papcon', 4210177915),
    ('passat', 4306419836),
    ('pcc', 4306420716),
    ('pccga', 4306420662),
    ('pcee', 7407085981),
    ('pcicon', 4210220794),
    ('pesos', 4306420731),
    ('pesw', 4306497572),
    ('pet', 7407104774),
    ('pica', 7407086728),
    ('pimrc', 4363605306),
    ('plans', 4393918190),
    ('powercon', 4363607772),
    ('powertech', 4306418811),
    ('precede', 4363608089),
    ('prrs', 7407106325),
    ('psec', 4210213236),
    ('pvsc-vol', 4210168774),
    ('qpain', 7407096833),
    ('raee', 7407106324),
    ('ramech', 7407087919),
    ('rams', 4210171523),
    ('rapcon', 7407120760),
    ('rawcon', 7407087810),
    ('redw', 7407087333),
    ('relaw', 4306424758),
    ('reldis', 4210201925),
    ('relphy', 4210238090),
    ('rfic', 4210202669),
    ('rfm', 4306418747),
    ('robionetics', 7407107709),
    ('robot', 4210217939),
    ('roman', 4393917971),
    ('rsp', 4210222258),
    ('rtcsa', 7407086768),
    ('rttas', 4210197441),
    ('sadfe', 7407107048),
    ('sbec', 4393916994),
    ('sbrn', 4210214732),
    ('scam', 7407087077),
    ('scft', 4306418850),
    ('scvt', 7407088501),
    ('sdf', 7407086235),
    ('seconw', 7407086276),
    ('seep', 7407088205),
    ('sensor', 4306419851),
    ('sew', 4210191939),
    ('sfcs', 4210175768),
    ('sharc', 7407106807),
    ('sibgra', 4210196976),
    ('siphotonics', 7407089019),
    ('sips', 4210209284),
    ('sispad', 4306419828),
    ('smi', 4306422697),
    ('smic', 7407088454),
    ('smicnd', 4210170846),
    ('spdp', 7407087759),
    ('spire', 4306420945),
    ('srdsw', 7407106055),
    ('ssap', 4306530856),
    ('ssdm', 4210224189),
    ('ssst', 4210172477),
    ('stherm', 4210186239),
    ('swste', 4306425424),
    ('taic-part', 7407105225),
    ('tdcllm', 7407113360),
    ('test', 4210209697),
    ('texcon', 7407106521),
    ('tfsa', 4306418823),
    ('time', 4306420224),
    ('tools', 4306421034),
    ('tron', 7407105339),
    ('ttm', 7407107271),
    ('tutcas', 4393914142),
    ('ubmk', 4306498178),
    ('ukricis', 7407088467),
    ('uksim', 7407088653),
    ('uwbst', 4306418610),
    ('vds', 7407106178),
    ('vecims', 4210169068),
    ('vetecf', 4210185404),
    ('vetecs', 4210185404),
    ('visual', 4393916909),
    ('viuf', 7407114004),
    ('vlsit', 4306421008),
    ('vtc', 4210185404),
    ('vtest', 4210180668),
    ('vts', 4210180668),
    ('vtsa', 7407086210),
    ('wcica', 4306421325),
    ('wcnc', 4210195247),
    ('wcncw', 7407087893),
    ('wcsn', 4306497913),
    ('wecwis', 4210186972),
    ('werc', 7407105472),
    ('wicsa', 7407086121),
    ('wkdd', 4306421263),
    ('wmcsa', 4210191346),
    ('wnyipw', 7407088793),
    ('wocn', 7407088009),
    ('wpdrts', 7407114855),
    ('wpmc', 4363607502),
    ('wsa', 4306419998),
    ('wsc', 4306421180),
    ('wse', 4210178652),
    ('ysf', 7407104862)
  ) AS m(acr, source_id)
  JOIN openalex.sources.sources s ON s.id = m.source_id AND s.type = 'conference'
) AS source
ON target.source_id IS NULL
   AND target.provenance = 'crossref'
   AND LOWER(target.raw_type) = 'proceedings-article'
   AND target.best_doi LIKE '10.1109/%'
   AND NOT exists(target.ids, x -> x.namespace LIKE '%issn%')
   AND regexp_extract(LOWER(target.best_doi), '^10\\.1109/([a-z]+(-[a-z]+)*)', 1) = source.acr
WHEN MATCHED THEN
  UPDATE SET target.source_id = source.source_id, target.is_oa_source = source.is_oa_source;

### Book parts typed 'other' and Oxford's glued publisher strings: the publisher's ebook platform (fills what every tier above left empty)

In [0]:
-- Late book tier (oxjob #1528). Book content the ebook-platform tier misses goes to its publisher's ebook platform
-- (oxjob #1528 POLICY.md, 2026-10-03: a chapter's source, and a whole book's, is the book series when it has one, else the
-- publisher's ebook platform; the platform's publisher is the imprint). Two kinds, 1.06M with no source on 3 Nov 2026:
-- (1) other_book_part: Crossref items typed 'other' that are book parts. Wiley (0.50M), Cambridge, Bristol UP, Vandenhoeck
--     & Ruprecht, World Scientific and others deposit chapters as 'other', which cell 3 gives no normalized_book_publisher.
--     Same platform key as the ebook-platform tier (unique keys only).
-- (2) oup_glued: Oxford University Press book content whose publisher string has the city glued on at ingest
--     ('Oxford University PressOxford', '...PressNew York, NY'; 0.35M) -> Oxford University Press eBooks.
-- Out: reference works, until their policy is set (Jason): containers whose name reads as a reference work
-- (encyclopedia, dictionary, lexicon, atlas, bibliography ...), containers with more than 200 Crossref book/'other' items,
-- Bloomsbury Fashion Central runway collections. Out after the bar: Bloomsbury's DOI prefix 10.5040 (with its imprints
-- Rowman & Littlefield, Praeger, Lexington, Greenwood; its videos, images and databases are typed 'other' too) and OECD
-- 10.1787 (figure and table DOIs). Bar: pre-registered 07:35 CST 3 Nov
-- (#1528 PLAN.md § Late book tier), Opus 5.5 on Crossref evidence, 150 per stratum: Wiley 149, other publishers 146 (145/145
-- without OECD), Oxford glued 149; Bloomsbury 10.5040 142 (failed, out); 10/10 known-bad controls caught.
-- Fills NULL sources only and runs after every other tier. Never touches a deposit that carries an ISSN.
-- Revert: delete this cell.
MERGE INTO openalex.works.locations_w_sources AS target
USING (WITH ebook_platform_titles AS (
  SELECT id AS source_id,
    LOWER(REGEXP_REPLACE(REGEXP_REPLACE(t, '(?i)\\s*ebooks$', ''), '[^\\p{L}\\p{N}]', '')) AS k
  FROM openalex.sources.sources
  LATERAL VIEW explode(concat(array(display_name), coalesce(alternate_titles, array()))) x AS t
  WHERE type = 'ebook platform' AND t IS NOT NULL
),
ebook_platform_keys AS (
  SELECT k, MIN(source_id) AS source_id FROM ebook_platform_titles
  WHERE length(k) >= 3 GROUP BY k HAVING COUNT(DISTINCT source_id) = 1
),
containers AS (
  SELECT normalized_source_name AS nsn, LOWER(REGEXP_REPLACE(publisher, '[^\\p{L}\\p{N}]', '')) AS npub,
    MAX(source_name) AS name, COUNT(*) AS parts,
    MAX(CASE WHEN LOWER(raw_type) = 'other' THEN 1 ELSE 0 END) AS has_other
  FROM openalex.works.locations_w_sources
  WHERE provenance = 'crossref' AND publisher IS NOT NULL AND normalized_source_name IS NOT NULL
    AND length(normalized_source_name) > 0
    AND (LOWER(raw_type) = 'other' OR LOWER(raw_type) IN ('book', 'books', 'book-chapter', 'book chapter', 'bookchapter', 'ebook chapter', 'book-part', 'book part', 'bookpart', 'book section', 'book-section', 'book article', 'chapter', 'inbook', 'monograph', 'printed monograph', 'edited-book', 'reference-book', 'book-set', 'libros', 'libro', 'info:eu-repo/semantics/book', 'info:eu-repo/semantics/bookpart'))
  GROUP BY 1, 2
)
SELECT c.nsn, c.npub, CASE WHEN c.npub RLIKE '^oxforduniversitypress(oxford|newyork|newyorkny|newyorkoxford|delhi|unitedkingdom|unitedstatesofamerica|ontario|toronto)$' THEN 4306463708 ELSE k.source_id END AS source_id,
  CASE WHEN c.npub RLIKE '^oxforduniversitypress(oxford|newyork|newyorkny|newyorkoxford|delhi|unitedkingdom|unitedstatesofamerica|ontario|toronto)$' THEN 'oup_glued' ELSE 'other_book_part' END AS rule
FROM containers c
LEFT JOIN ebook_platform_keys k ON k.k = c.npub
WHERE c.parts <= 200
  AND NOT c.name RLIKE '(?i)(encyclop|enzyklop|dictionar|diccionari|dictionnaire|w[öo]rterbuch|lexikon|lexicon|glossar|thesaurus|almanac|gazetteer|who.?s who|biographical|bibliograph|chronolog|\\bindex of\\b|catalog(ue)? of|\\batlas\\b|\\bdirectory\\b|\\bdatabook\\b|data book|compendium|reference guide|a to z|a-z\\b)'
  AND NOT c.name RLIKE '(?i)(spring/summer|autumn/winter|fall/winter|pre-fall|resort [0-9]{4}|cruise [0-9]{4}|spring [0-9]{4}|fall [0-9]{4})'
  AND (c.npub RLIKE '^oxforduniversitypress(oxford|newyork|newyorkny|newyorkoxford|delhi|unitedkingdom|unitedstatesofamerica|ontario|toronto)$' OR (k.source_id IS NOT NULL AND c.has_other = 1))
) AS source
ON target.source_id IS NULL
   AND target.provenance = 'crossref'
   AND NOT exists(target.ids, x -> lower(x.namespace) LIKE '%issn%')
   AND target.best_doi NOT LIKE '10.5040/%'
   AND target.best_doi NOT LIKE '10.1787/%'
   AND target.normalized_source_name = source.nsn
   AND LOWER(REGEXP_REPLACE(target.publisher, '[^\\p{L}\\p{N}]', '')) = source.npub
   AND (LOWER(target.raw_type) = 'other'
        OR (source.rule = 'oup_glued' AND LOWER(target.raw_type) IN ('book', 'books', 'book-chapter', 'book chapter', 'bookchapter', 'ebook chapter', 'book-part', 'book part', 'bookpart', 'book section', 'book-section', 'book article', 'chapter', 'inbook', 'monograph', 'printed monograph', 'edited-book', 'reference-book', 'book-set', 'libros', 'libro', 'info:eu-repo/semantics/book', 'info:eu-repo/semantics/bookpart')))
WHEN MATCHED THEN
  UPDATE SET target.source_id = source.source_id;

-- Late book tier, reference-work pass (oxjob #1528). Jason, 3 Nov 2026: reference works follow the book rule, so an
-- encyclopedia or dictionary entry goes on its publisher's ebook platform like a chapter (#1528 POLICY.md § Decided).
-- Runs after the statement above and fills what it left: Crossref 'other' or 'reference-entry' items with no source and no
-- ISSN whose publisher is a unique platform key (or Oxford's glued strings), kept only when the item's ISBN is also on a
-- Crossref book record (book, monograph, edited-book, reference-book, book-set), which keeps out the videos, images and
-- database records some publishers deposit as 'other'. OECD 10.1787 figures and runway collections stay out. 461K works
-- on 3 Nov 2026 (Bloomsbury and its imprints 395K). Bar: pre-registered 12:50 CST 3 Nov, 150 per stratum, Bloomsbury
-- family 150, other publishers 150, reference-entry 150; 9/9 known-bad controls caught. Revert: delete this statement.
MERGE INTO openalex.works.locations_w_sources AS target
USING (WITH ebook_platform_titles AS (
  SELECT id AS source_id,
    LOWER(REGEXP_REPLACE(REGEXP_REPLACE(t, '(?i)\\s*ebooks$', ''), '[^\\p{L}\\p{N}]', '')) AS k
  FROM openalex.sources.sources
  LATERAL VIEW explode(concat(array(display_name), coalesce(alternate_titles, array()))) x AS t
  WHERE type = 'ebook platform' AND t IS NOT NULL
),
ebook_platform_keys AS (
  SELECT k, MIN(source_id) AS source_id FROM ebook_platform_titles
  WHERE length(k) >= 3 GROUP BY k HAVING COUNT(DISTINCT source_id) = 1
),
book_isbns AS (
  SELECT DISTINCT regexp_replace(x.id, '[^0-9Xx]', '') AS isbn
  FROM openalex.works.locations_w_sources b LATERAL VIEW explode(b.ids) e AS x
  WHERE b.provenance = 'crossref'
    AND LOWER(b.raw_type) IN ('book', 'monograph', 'edited-book', 'reference-book', 'book-set')
    AND lower(x.namespace) LIKE '%isbn%'
),
item_isbns AS (
  SELECT t.best_doi, LOWER(REGEXP_REPLACE(t.publisher, '[^\\p{L}\\p{N}]', '')) AS npub,
    regexp_replace(x.id, '[^0-9Xx]', '') AS isbn
  FROM openalex.works.locations_w_sources t LATERAL VIEW explode(t.ids) e AS x
  WHERE t.provenance = 'crossref' AND t.source_id IS NULL
    AND LOWER(t.raw_type) IN ('other', 'reference-entry')
    AND t.publisher IS NOT NULL AND t.source_name IS NOT NULL
    AND t.best_doi NOT LIKE '10.1787/%'
    AND NOT t.source_name RLIKE '(?i)(spring/summer|autumn/winter|fall/winter|pre-fall|resort [0-9]{4}|cruise [0-9]{4}|spring [0-9]{4}|fall [0-9]{4})'
    AND NOT exists(t.ids, y -> lower(y.namespace) LIKE '%issn%')
    AND lower(x.namespace) LIKE '%isbn%'
)
SELECT i.best_doi,
  MIN(CASE WHEN i.npub RLIKE '^oxforduniversitypress(oxford|newyork|newyorkny|newyorkoxford|delhi|unitedkingdom|unitedstatesofamerica|ontario|toronto)$' THEN 4306463708 ELSE k.source_id END) AS source_id
FROM item_isbns i
JOIN book_isbns bi ON bi.isbn = i.isbn
LEFT JOIN ebook_platform_keys k ON k.k = i.npub
WHERE i.npub RLIKE '^oxforduniversitypress(oxford|newyork|newyorkny|newyorkoxford|delhi|unitedkingdom|unitedstatesofamerica|ontario|toronto)$' OR k.source_id IS NOT NULL
GROUP BY i.best_doi
) AS source
ON target.best_doi = source.best_doi
   AND target.source_id IS NULL
   AND target.provenance = 'crossref'
   AND LOWER(target.raw_type) IN ('other', 'reference-entry')
WHEN MATCHED THEN
  UPDATE SET target.source_id = source.source_id;

### Reference works and books on journal-typed records with no ISSN: move to the publisher's ebook platform (after every tier)

In [0]:
-- Reference-work move (oxjob #1528). Reference works and books are never sources of their own (oxjob #1528 POLICY.md,
-- 2026-10-03: a chapter's source is the book series when it has one, else the publisher's ebook platform; 2026-11-03,
-- Jason: reference works follow the book rule). The name tier (cell 1) still puts an entry whose container is "Who Was
-- Who" on a journal-typed record called "Who Was Who": such records have no ISSN, so the #828 book guard lets them
-- through, and every book tier above fills NULL only. 1.26M Crossref entries on 1,013 such records on 4 Nov 2026
-- (Benezit Dictionary of Artists, Oxford Dictionary of National Biography, Oxford Music Online, SpringerReference,
-- Encyclopedia of Life Sciences, Who's Who, Birds of the World ...).
-- Moves a location to its publisher's ebook platform when: its source is a journal-typed record with no ISSN; the
-- deposit has no ISSN and Crossref types it as a book part, 'other' or 'reference-entry'; its publisher is a unique
-- ebook-platform key (the ebook-platform tier's key) or one of Oxford's glued strings (-> Oxford University Press
-- eBooks); and that record's items of this class carry at most 10 distinct ISBNs. More ISBNs = a book series typed
-- journal ("Review of Progress in Quantitative Nondestructive Evaluation") or a title many books share ("Epilepsy"):
-- those stay. Publishers with no platform (Qeios "Definitions", Radiopaedia.org) stay too.
-- Bar: pre-registered 16:45 CST 4 Nov 2026 (oxjob #1528 PLAN.md § Reference-work move), Opus 5.5 rubric refmove-v1 on
-- the Crossref item, its book records, the record it sits on and the platform, 150 per stratum, Wilson LB >= 0.93:
-- reference-entry 150/150, other 150/150, book-part types 149/150, records with < 500 moved items 147/150 (2 of 3
-- misses OECD figures, so 10.1787 is out); 10/10 known-bad controls caught.
-- Runs after every other tier. MOVES rows that have a source (only rows on journal-typed records with no ISSN).
-- Revert: delete this cell.
MERGE INTO openalex.works.locations_w_sources AS target
USING (WITH ebook_platform_titles AS (
  SELECT id AS source_id,
    LOWER(REGEXP_REPLACE(REGEXP_REPLACE(t, '(?i)\\s*ebooks$', ''), '[^\\p{L}\\p{N}]', '')) AS k
  FROM openalex.sources.sources
  LATERAL VIEW explode(concat(array(display_name), coalesce(alternate_titles, array()))) x AS t
  WHERE type = 'ebook platform' AND t IS NOT NULL
),
ebook_platform_keys AS (
  SELECT k, MIN(source_id) AS source_id FROM ebook_platform_titles
  WHERE length(k) >= 3 GROUP BY k HAVING COUNT(DISTINCT source_id) = 1
),
journals_no_issn AS (
  SELECT id FROM openalex.sources.sources WHERE type = 'journal' AND coalesce(size(issns), 0) = 0
),
items AS (
  SELECT t.source_id AS record_id, LOWER(REGEXP_REPLACE(t.publisher, '[^\\p{L}\\p{N}]', '')) AS npub, t.ids
  FROM openalex.works.locations_w_sources t JOIN journals_no_issn j ON j.id = t.source_id
  WHERE t.provenance = 'crossref' AND t.publisher IS NOT NULL
    AND NOT exists(t.ids, x -> lower(x.namespace) LIKE '%issn%')
    AND (LOWER(t.raw_type) IN ('other', 'reference-entry') OR LOWER(t.raw_type) IN ('book', 'books', 'book-chapter', 'book chapter', 'bookchapter', 'ebook chapter', 'book-part', 'book part', 'bookpart', 'book section', 'book-section', 'book article', 'chapter', 'inbook', 'monograph', 'printed monograph', 'edited-book', 'reference-book', 'book-set', 'libros', 'libro', 'info:eu-repo/semantics/book', 'info:eu-repo/semantics/bookpart'))
),
record_isbns AS (
  SELECT record_id, count(DISTINCT regexp_replace(x.id, '[^0-9Xx]', '')) AS n_isbn
  FROM items LATERAL VIEW explode(ids) e AS x
  WHERE lower(x.namespace) LIKE '%isbn%'
  GROUP BY record_id
),
pairs AS (SELECT DISTINCT record_id, npub FROM items)
SELECT p.record_id, p.npub,
  CASE WHEN p.npub RLIKE '^oxforduniversitypress(oxford|newyork|newyorkny|newyorkoxford|delhi|unitedkingdom|unitedstatesofamerica|ontario|toronto)$' THEN 4306463708 ELSE k.source_id END AS platform_id
FROM pairs p
LEFT JOIN ebook_platform_keys k ON k.k = p.npub
LEFT JOIN record_isbns r ON r.record_id = p.record_id
WHERE coalesce(r.n_isbn, 0) <= 10
  AND (p.npub RLIKE '^oxforduniversitypress(oxford|newyork|newyorkny|newyorkoxford|delhi|unitedkingdom|unitedstatesofamerica|ontario|toronto)$' OR k.source_id IS NOT NULL)
) AS source
ON target.source_id = source.record_id
   AND target.provenance = 'crossref'
   AND NOT exists(target.ids, x -> lower(x.namespace) LIKE '%issn%')
   AND target.best_doi NOT LIKE '10.1787/%'
   AND (LOWER(target.raw_type) IN ('other', 'reference-entry') OR LOWER(target.raw_type) IN ('book', 'books', 'book-chapter', 'book chapter', 'bookchapter', 'ebook chapter', 'book-part', 'book part', 'bookpart', 'book section', 'book-section', 'book article', 'chapter', 'inbook', 'monograph', 'printed monograph', 'edited-book', 'reference-book', 'book-set', 'libros', 'libro', 'info:eu-repo/semantics/book', 'info:eu-repo/semantics/bookpart'))
   AND LOWER(REGEXP_REPLACE(target.publisher, '[^\\p{L}\\p{N}]', '')) = source.npub
WHEN MATCHED THEN
  UPDATE SET target.source_id = source.platform_id;

### DTIC reports and university theses: the registrant's repository, by DOI prefix (fills what every tier above left empty)

In [0]:
-- Report and thesis registrants (oxjob #1528). A technical report deposited by DTIC, or a thesis deposited by a university
-- thesis library, has the sponsor or the university as its Crossref container ("US Dept of the Army", "Universidade de Sao
-- Paulo", "Aix-Marseille"), so no tier above gives it a source. Its source is the registrant's repository (Jason,
-- 2026-07-20, oxjob #652, now #1528: DTIC and university thesis libraries are match targets for their reports and
-- theses). Keyed on the DOI prefix, which belongs to that registrant alone: 10.21236 reports -> Defense Technical
-- Information Center (DTIC); 10.11606 theses (d./t. DOIs only; the numeric ones belong to another USP library) -> Digital
-- Library of Theses and Dissertations (Universidade de Sao Paulo); 10.70675 theses -> theses.fr (ABES). ~487K locations
-- on 5 Nov 2026. Bar: census of registered DOI URLs, 150 per prefix (pre-registered 5 Nov 13:51 CST): DTIC 150/150 on dtic.mil, USP 146/150 on teses.usp.br (the 4 misses were the numeric DOIs now left out), theses.fr 150/150.
-- Fills NULL sources only, after every other tier. Revert: delete this cell.
MERGE INTO openalex.works.locations_w_sources AS target
USING (SELECT r.prefix, r.raw_type, r.doi_pattern, r.source_id
  FROM VALUES ('10.21236', 'report', '^10\\.21236/', 4406923043), ('10.11606', 'dissertation', '(?i)^10\\.11606/[dt]\\.', 4377196381), ('10.70675', 'dissertation', '^10\\.70675/', 4406923047) AS r(prefix, raw_type, doi_pattern, source_id)
  JOIN openalex.sources.sources s ON s.id = r.source_id
) AS source
ON target.source_id IS NULL
   AND target.provenance = 'crossref'
   AND target.best_doi LIKE concat(source.prefix, '/%')
   AND target.best_doi RLIKE source.doi_pattern
   AND LOWER(target.raw_type) = source.raw_type
   AND NOT exists(target.ids, x -> lower(x.namespace) LIKE '%issn%')
WHEN MATCHED THEN
  UPDATE SET target.source_id = source.source_id;

### OSTI reports, BHL scans, Spreng's dictionary: the hosting record, by DOI prefix (fills what every tier above left empty)

In [0]:
-- Registrant tier, wave 2 (oxjob #1528). Items a registrant deposits under its own DOI prefix with no container that any
-- tier above can key: their source is the record that hosts them, keyed on the DOI prefix + DOI shape + Crossref type.
--   10.2172 reports -> OSTI OAI (DOE's Office of Scientific and Technical Information; the container is the lab or office).
--   10.5962/bhl.title. monographs -> Biodiversity Heritage Library (BHL's scanned titles; no container).
--   10.24894/SprengGlossarium_ datasets -> Schwabe Verlag eBooks (entries of Spreng's Allgemeines deutsches Glossarium, a
--     dictionary; reference works follow the book rule: the publisher's ebook platform, POLICY.md 2026-11-03).
-- Policy: Jason 2026-07-20 (#652, now #1528: aggregators like DTIC and OSTI and registrants' repositories are match
-- targets) and POLICY.md 2026-11-03. ~541K locations on 6 Nov 2026. Bar: the URL each registrant registered for the DOI (doi.org handle API), 150 per target, pre-registered 09:51 CST 6 Nov: OSTI 150/150, BHL 150/150, Schwabe 150/150 (OSF preprint servers failed and stay out).
-- Fills NULL sources only, after every other tier. Revert: delete this cell.
MERGE INTO openalex.works.locations_w_sources AS target
USING (SELECT r.prefix, r.raw_type, r.doi_pattern, r.source_id
  FROM VALUES ('10.2172', 'report', '^10\\.2172/', 4306402487), ('10.5962', 'monograph', '^10\\.5962/bhl\\.title\\.', 4306402618), ('10.24894', 'dataset', '(?i)^10\\.24894/sprengglossarium_', 4306479414) AS r(prefix, raw_type, doi_pattern, source_id)
  JOIN openalex.sources.sources s ON s.id = r.source_id
) AS source
ON target.source_id IS NULL
   AND target.provenance = 'crossref'
   AND target.best_doi LIKE concat(source.prefix, '/%')
   AND target.best_doi RLIKE source.doi_pattern
   AND LOWER(target.raw_type) = source.raw_type
   AND NOT exists(target.ids, x -> lower(x.namespace) LIKE '%issn%')
WHEN MATCHED THEN
  UPDATE SET target.source_id = source.source_id;

### EGU General Assembly abstracts: the conference series, by DOI shape (fills what every tier above left empty)

In [0]:
-- EGU General Assembly abstracts (oxjob #1528). Copernicus deposits each EGU General Assembly abstract as Crossref
-- posted-content with a DOI 10.5194/egusphere-eguNN-N and no container, so no tier above keys it. Its source is the
-- conference series, EGU General Assembly Conference Abstracts (conferences: the series is the source, POLICY.md
-- 2026-10-03). ~104K locations on 6 Nov 2026. Bar: the URL Copernicus registered for each DOI (doi.org handle API) is the abstract's page on meetingorganizer.copernicus.org under that year's General Assembly (/EGUyy/), 150/150, pre-registered 10:09 CST 6 Nov.
-- Fills NULL sources only, after every other tier. Revert: delete this cell.
MERGE INTO openalex.works.locations_w_sources AS target
USING (SELECT id AS source_id FROM openalex.sources.sources WHERE id = 4306508951) AS source
ON target.source_id IS NULL
   AND target.provenance = 'crossref'
   AND LOWER(target.raw_type) = 'posted-content'
   AND target.best_doi LIKE '10.5194/egusphere-egu%'
   AND target.best_doi RLIKE '^10\\.5194/egusphere-egu[0-9]{2}-[0-9]+$'
   AND NOT exists(target.ids, x -> lower(x.namespace) LIKE '%issn%')
WHEN MATCHED THEN
  UPDATE SET target.source_id = source.source_id;

### Bloomsbury-family book parts: the imprint's ebook platform, by ISBN registrant block

In [0]:
-- Bloomsbury-family book parts by ISBN block (oxjob #1528). Bloomsbury and the presses it owns or distributes deposit book chapters as
-- Crossref 'other' with DOI 10.5040/<ISBN-13>.<part>, often naming only the parent company ('The Rowman & Littlefield
-- Publishing Group'), so no publisher key finds the imprint, and the late book tier leaves 10.5040 out. Here only parts of
-- real books (the DOI's ISBN is on a Crossref book record). The imprint comes from the book's ISBN registrant blocks:
-- openalex.sources.isbn_block_platforms maps a block to the platform of the one imprint it belongs to (judged per block; shared ebook blocks are
-- left out), and a row takes the platform only when all its mapped blocks agree. Source = the imprint's ebook platform
-- (POLICY.md 2026-10-03; reference works too, 2026-11-03). ~87K locations on 6 Nov 2026. Bar: the item bar, 150 drawn rows judged by Opus 5.5 (part of a book AND the platform of the book's publisher or imprint): 147/150, Wilson LB 0.943, pre-registered 10:55 CST 6 Nov; blocks judged by Opus 5.5 + Sol, both agreeing, controls 0 of 3.
-- Fills NULL sources only, after every other tier. Revert: delete this cell (and drop openalex.sources.isbn_block_platforms).
MERGE INTO openalex.works.locations_w_sources AS target
USING (
  WITH book_isbns AS (
  SELECT DISTINCT regexp_replace(x.id, '[^0-9Xx]', '') AS isbn
  FROM openalex.works.locations_w_sources b LATERAL VIEW explode(b.ids) e AS x
  WHERE b.provenance = 'crossref' AND lower(b.raw_type) IN ('book', 'monograph', 'edited-book', 'reference-book', 'book-set')
    AND lower(x.namespace) LIKE '%isbn%'),
pop AS (
  SELECT t.publisher, t.source_name, t.best_doi, regexp_extract(t.best_doi, '^10\\.5040/(97[89][0-9]{10})[.-]', 1) isbn
  FROM openalex.works.locations_w_sources t
  WHERE t.provenance = 'crossref' AND t.source_id IS NULL AND lower(t.raw_type) = 'other'
    AND t.best_doi RLIKE '^10\\.5040/97[89][0-9]{10}[.-]'
    AND NOT exists(t.ids, y -> lower(y.namespace) LIKE '%issn%')
    AND regexp_extract(t.best_doi, '^10\\.5040/(97[89][0-9]{10})[.-]', 1) IN (SELECT isbn FROM book_isbns)),
  pop_dois AS (SELECT DISTINCT best_doi FROM pop),
  it0 AS (
    SELECT l.provenance, l.native_id, l.best_doi, regexp_replace(i.id, '[^0-9Xx]', '') d
    FROM openalex.works.locations_w_sources l JOIN pop_dois p ON p.best_doi = l.best_doi LATERAL VIEW explode(l.ids) e AS i
    WHERE l.provenance = 'crossref' AND l.source_id IS NULL AND lower(l.raw_type) = 'other' AND lower(i.namespace) LIKE '%isbn%'
    UNION ALL
    SELECT l.provenance, l.native_id, l.best_doi, regexp_extract(l.best_doi, '^10\\.5040/(97[89][0-9]{10})[.-]', 1)
    FROM openalex.works.locations_w_sources l JOIN pop_dois p ON p.best_doi = l.best_doi
    WHERE l.provenance = 'crossref' AND l.source_id IS NULL AND lower(l.raw_type) = 'other'),
  it AS (SELECT DISTINCT provenance, native_id, best_doi, CASE WHEN length(d) = 13 THEN substr(d, 1, 12) WHEN length(d) = 10 THEN concat('978', substr(d, 1, 9)) END body FROM it0),
  blk AS (SELECT block, source_id FROM openalex.sources.isbn_block_platforms),
  hit AS (SELECT it.provenance, it.native_id, it.best_doi, blk.source_id FROM it JOIN blk ON it.body LIKE concat(blk.block, '%')),
  tg AS (SELECT provenance, native_id, best_doi, min(source_id) source_id FROM hit GROUP BY 1, 2, 3
         HAVING count(DISTINCT source_id) = 1)
  SELECT provenance, native_id, source_id FROM tg
) AS source
ON target.provenance = source.provenance AND target.native_id = source.native_id AND target.source_id IS NULL
WHEN MATCHED THEN
  UPDATE SET target.source_id = source.source_id;

### Standards bodies and The Conversation: the publisher's collection, by DOI prefix (fills what every tier above left empty)

In [0]:
-- Standards bodies and The Conversation (oxjob #1528). Items with no venue get their publisher's collection as source (Jason,
-- 2026-11-06): each standards body's standards go on its standards record (BSI Standards, ASTM Standards, DIN Standards,
-- SAE Standards, USP-NF, IEEE Standards, SMPTE Standards, AWWA Standards, AMPP Standards), and The Conversation's articles
-- on The Conversation. Keyed on the DOI prefix, which belongs to that body, and the Crossref type ('standard';
-- 'posted-content' for The Conversation). ~647K locations on 6 Nov 2026. Bar: the URL each body registered for the DOI (doi.org handle API) is on that body's site, 150 per stratum, pre-registered 18:19 CST 6 Nov: BSI, ASTM, DIN, SAE, USP, The Conversation and the four small bodies pooled, each 150/150.
-- Fills NULL sources only, after every other tier. Revert: delete this cell.
MERGE INTO openalex.works.locations_w_sources AS target
USING (SELECT r.body, r.prefix, r.raw_type, r.doi_pattern, r.source_id
  FROM VALUES ('BSI', '10.3403', 'standard', '^10\\.3403/', 7407122196), ('ASTM', '10.1520', 'standard', '^10\\.1520/', 7407122197), ('DIN', '10.31030', 'standard', '^10\\.31030/', 7407122199), ('SAE', '10.4271', 'standard', '^10\\.4271/', 7407122200), ('USP', '10.31003', 'standard', '^10\\.31003/', 7407122201), ('IEEE', '10.1109', 'standard', '^10\\.1109/', 7407122202), ('SMPTE', '10.5594', 'standard', '^10\\.5594/', 7407122198), ('AWWA', '10.12999', 'standard', '^10\\.12999/', 7407122203), ('AMPP', '10.5006', 'standard', '^10\\.5006/', 7407122204), ('Conversation', '10.64628', 'posted-content', '^10\\.64628/', 2764582943) AS r(body, prefix, raw_type, doi_pattern, source_id)
  JOIN openalex.sources.sources s ON s.id = r.source_id
) AS source
ON target.source_id IS NULL
   AND target.provenance = 'crossref'
   AND target.best_doi LIKE concat(source.prefix, '/%')
   AND target.best_doi RLIKE source.doi_pattern
   AND LOWER(target.raw_type) = source.raw_type
   AND NOT exists(target.ids, x -> lower(x.namespace) LIKE '%issn%')
WHEN MATCHED THEN
  UPDATE SET target.source_id = source.source_id;

### OSF preprint servers: the server, by DOI prefix (fills what every tier above left empty)

In [0]:
-- OSF preprint servers (oxjob #1528). OSF mints each preprint server's DOIs under the server's own Crossref prefix
-- (PsyArXiv 10.31234, SocArXiv 10.31235, OSF Preprints 10.31219 ...) and deposits them as posted-content with no container,
-- so no tier above keys them. Their source is the server (preprint servers are sources, Jason 2026-07-20). A preprint OSF
-- later removed keeps its server; removing such works is a separate cleanup. ~236K locations on 7 Nov 2026. Bar: pre-registered 2026-11-07 08:06 CST (oxjobs a41febc57), registered DOI URL on osf.io and the server named by OSF's record (live) or Crossref's deposit (removed): OSF Preprints 149/150, other servers 147/150.
-- Fills NULL sources only, after every other tier. Revert: delete this cell.
MERGE INTO openalex.works.locations_w_sources AS target
USING (SELECT r.prefix, r.doi_pattern, r.source_id
  FROM VALUES ('10.31219', '^10\\.31219/osf\\.io/', 4306401127), ('10.31234', '^10\\.31234/osf\\.io/', 4306401687), ('10.31235', '^10\\.31235/osf\\.io/', 4306401238), ('10.31227', '^10\\.31227/osf\\.io/', 4306402438), ('10.31237', '^10\\.31237/osf\\.io/', 4306401491), ('10.35542', '^10\\.35542/osf\\.io/', 4306402530), ('10.31228', '^10\\.31228/osf\\.io/', 4306401405), ('10.31222', '^10\\.31222/osf\\.io/', 4306402619), ('10.33767', '^10\\.33767/osf\\.io/', 4306402499), ('10.31230', '^10\\.31230/osf\\.io/', 4306401713), ('10.31221', '^10\\.31221/osf\\.io/', 4306400047), ('10.31730', '^10\\.31730/osf\\.io/', 4306402219), ('10.31220', '^10\\.31220/osf\\.io/', 4306400402), ('10.31229', '^10\\.31229/osf\\.io/', 4306400670), ('10.31236', '^10\\.31236/osf\\.io/', 4306402352), ('10.1149', '^10\\.1149/osf\\.io/', 4306402531), ('10.31231', '^10\\.31231/osf\\.io/', 4306401745), ('10.31233', '^10\\.31233/osf\\.io/', 4306400732), ('10.34055', '^10\\.34055/osf\\.io/', 4306402544), ('10.31226', '^10\\.31226/osf\\.io/', 4306401712), ('10.35543', '^10\\.35543/osf\\.io/', 4306402473), ('10.31225', '^10\\.31225/osf\\.io/', 4306401693), ('10.31232', '^10\\.31232/osf\\.io/', 4306401186) AS r(prefix, doi_pattern, source_id)
  JOIN openalex.sources.sources s ON s.id = r.source_id
) AS source
ON target.source_id IS NULL
   AND target.provenance = 'crossref'
   AND LOWER(target.raw_type) = 'posted-content'
   AND target.best_doi LIKE concat(source.prefix, '/osf.io/%')
   AND target.best_doi RLIKE source.doi_pattern
   AND NOT exists(target.ids, x -> lower(x.namespace) LIKE '%issn%')
WHEN MATCHED THEN
  UPDATE SET target.source_id = source.source_id;

In [0]:
SELECT * FROM openalex.works.locations_w_sources